In [ ]:
# --- Sistema y Google Drive ---
import os
import glob
from google.colab import drive

# --- Procesamiento de Datos y Evaluación ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# --- Procesamiento de Imágenes y Visión Artificial ---
import cv2
from skimage import io, data
from skimage.color import rgb2gray
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# --- Visualización ---
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# --- Deep Learning (TensorFlow y Keras) ---
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.utils import plot_model, model_to_dot
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten, Conv2D, 
    MaxPool2D, BatchNormalization, Conv2DTranspose, 
    Input, Activation, concatenate
)

In [ ]:
# Definición de directorios
dataset_dir = "/content/drive/MyDrive/New_U_Net/RGB"

train_img_dir = f"{dataset_dir}/Set_A_img"
train_mask_dir = f"{dataset_dir}/Set_A_mask"
test_img_dir  = f"{dataset_dir}/Set_B_img"
test_mask_dir = f"{dataset_dir}/Set_B_mask"

# Obtención y ordenamiento de rutas de archivos
train_img_paths = sorted(glob.glob(train_img_dir + "/*.jpg"))
train_mask_paths = sorted(glob.glob(train_mask_dir + "/*.png"))

test_img_paths = sorted(glob.glob(test_img_dir + "/*.jpg"))
test_mask_paths = sorted(glob.glob(test_mask_dir + "/*.png"))

# Visualización de conteo de archivos
print("Train images:", len(train_img_paths))
print("Train masks:", len(train_mask_paths))
print("Test images:", len(test_img_paths))
print("Test masks:", len(test_mask_paths))

In [ ]:
# Dimensiones de entrada 
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 4

In [ ]:
# Normalización de imagen-máscara
def load_image(img_path, mask_path):
    # Procesamiento de la imagen RGB
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_HEIGHT, IMG_WIDTH))
    img = tf.cast(img, tf.float32) / 255.0  # Normalización [0, 1]

    # Procesamiento de la máscara (Ground Truth)
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (IMG_HEIGHT, IMG_WIDTH))
    mask = tf.cast(mask > 127, tf.float32)  # Binarización a 0 y 1

    return img, mask

In [ ]:
# Indexacion y creacion 
train_img_paths = sorted(glob.glob(train_img_dir + "/*.jpg"))
train_mask_paths = sorted(glob.glob(train_mask_dir + "/*.png"))
test_img_paths = sorted(glob.glob(test_img_dir + "/*.jpg"))
test_mask_paths = sorted(glob.glob(test_mask_dir + "/*.png"))

print(f"Imágenes de entrenamiento encontradas: {len(train_img_paths)}")
print(f"Imágenes de prueba encontradas: {len(test_img_paths)}")

#  Creación del Dataset principal de entrenamiento
train_dataset = tf.data.Dataset.from_tensor_slices((train_img_paths, train_mask_paths))
train_dataset = train_dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

#  División en Entrenamiento (80%) y Validación (20%)
train_size = int(0.8 * len(train_img_paths))
val_size = len(train_img_paths) - train_size

train_ds = train_dataset.take(train_size).shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds   = train_dataset.skip(train_size).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

#  Creación del Dataset de prueba (Test)
test_ds = tf.data.Dataset.from_tensor_slices((test_img_paths, test_mask_paths))
test_ds = test_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("-" * 30)
print(f"Dataset de Entrenamiento (train_ds) listo: {train_size} muestras.")
print(f"Dataset de Validación (val_ds) listo:    {val_size} muestras.")
print(f"Dataset de Prueba (test_ds) listo:      {len(test_img_paths)} muestras.")

In [ ]:
# IoU
def iou_metric(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

# Dice Coefficient
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

# Sensibilidad (Recall)
def sensitivity(y_true, y_pred):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(tf.round(y_pred), [-1])
    TP = tf.reduce_sum(y_true_f * y_pred_f)
    FN = tf.reduce_sum(y_true_f * (1 - y_pred_f))
    return TP / (TP + FN + tf.keras.backend.epsilon())

# Especificidad 
def specificity(y_true, y_pred):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(tf.round(y_pred), [-1])
    TN = tf.reduce_sum((1 - y_true_f) * (1 - y_pred_f))
    FP = tf.reduce_sum((1 - y_true_f) * y_pred_f)
    return TN / (TN + FP + tf.keras.backend.epsilon())